In [ ]:
"""
import and config
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split


import os
import re
from typing import List
import struct

from torch.cuda import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau


# 设置设备
device = torch.device("cuda")
print("使用设备:", device)

folder_path = "./data/widar_data"
task_type = 'classification'
sub_type = 'Gesture Recognition'
final_fs = 1000
num_samples = 100000
batch_size = 32

使用设备: cuda


In [ ]:
"""
模型
"""


class AdaptiveCSIModel(nn.Module):
    def __init__(self, task_type='classification', num_classes=10, output_dim=1,
                 base_channels=32, latent_dim=128):
        super().__init__()
        self.task_type = task_type

        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.GELU(),
            nn.Conv2d(base_channels, base_channels*2, 3, groups=base_channels),
            nn.Conv2d(base_channels*2, base_channels*2, 1),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten()
        )

        self.temporal_processor = nn.Sequential(
            nn.Linear(base_channels*2*16, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.GELU(),
            nn.TransformerEncoderLayer(
                d_model=latent_dim,
                nhead=4,
                dim_feedforward=latent_dim*2,
                dropout=0.1,
                batch_first=True
            )
        )

        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()

        if task_type == 'classification':
            self.output_layer = nn.Linear(latent_dim, num_classes)
        elif task_type == 'regression':
            self.output_layer = nn.Sequential(
                nn.Linear(latent_dim, latent_dim//2),
                nn.GELU(),
                nn.Linear(latent_dim//2, output_dim)
            )
        else:
            raise ValueError("不支持的task_type: 必须是'classification'或'regression'")

    def forward(self, x):
        # 若输入是 [B, T, F, A, 2]，先合并最后一维
        if x.ndim == 5 and x.shape[-1] == 2:
            x = torch.norm(x, dim=-1)

        B, T, F, A = x.shape
        spatial_input = x.view(B*T, 1, F, A)
        spatial_feat = self.spatial_encoder(spatial_input)
        spatial_feat = spatial_feat.view(B, T, -1)

        temporal_feat = self.temporal_processor(spatial_feat)
        pooled = self.adaptive_pool(temporal_feat.transpose(1, 2))
        features = self.flatten(pooled)
        return self.output_layer(features)



In [ ]:
def train_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0
    samples_processed = 0

    # 根据模型的任务类型选择损失函数
    if model.task_type == 'classification':
        criterion = nn.CrossEntropyLoss()
    else:  # regression
        criterion = nn.MSELoss()

    for batch_data, batch_labels in loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)

        # 梯度重置
        optimizer.zero_grad()

        # 混合精度训练
        with amp.autocast():
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)

        # 反向传播
        scaler.scale(loss).backward()

        # 梯度裁剪防爆炸
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # 参数更新
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_data.size(0)
        samples_processed += batch_data.size(0)

    return total_loss / samples_processed if samples_processed > 0 else float('inf')

def evaluate(model, loader, device):
    model.eval()

    if model.task_type == 'classification':
        correct = 0
        total = 0
    else:  # regression
        all_preds = []
        all_targets = []

    with torch.no_grad():
        for batch_data, batch_labels in loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_data)

            if model.task_type == 'classification':
                _, predicted = torch.max(outputs, 1)
                total += batch_labels.size(0)
                correct += (predicted == batch_labels).sum().item()
            else:
                all_preds.append(outputs.cpu())
                all_targets.append(batch_labels.cpu())

    if model.task_type == 'classification':
        return correct / total if total > 0 else 0.0
    else:
        if all_preds:
            all_preds = torch.cat(all_preds)
            all_targets = torch.cat(all_targets)
            mse = F.mse_loss(all_preds, all_targets).item()
            mae = F.l1_loss(all_preds, all_targets).item()
            return {'mse': mse, 'mae': mae}
        return {'mse': float('inf'), 'mae': float('inf')}

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # 确保epochs是整数
    if not isinstance(epochs, int):
        try:
            epochs = int(epochs)
        except (ValueError, TypeError):
            raise TypeError("epochs参数必须是整数")

    # 优化器配置
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = amp.GradScaler()

    # 学习率调度器（根据任务类型调整）
    if model.task_type == 'classification':
        scheduler = ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3, verbose=True
        )
        metric_name = '准确率'
    else:
        scheduler = ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3, verbose=True
        )
        metric_name = 'MSE'

    best_metric = -float('inf') if model.task_type == 'classification' else float('inf')
    best_model_path = f'./sdp/results/saved_models/best_{model.task_type}_model.pth'
    epochs_since_improvement = 0

    for epoch in range(epochs):
        # 训练阶段
        train_loss = train_epoch(model, train_loader, optimizer, scaler, device)

        # 验证阶段
        if model.task_type == 'classification':
            val_metric = evaluate(model, val_loader, device)
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"训练损失: {train_loss:.4f} | 验证{metric_name}: {val_metric:.4f}")

            # 检查是否是最佳模型
            if val_metric > best_metric:
                best_metric = val_metric
                epochs_since_improvement = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"保存新的最佳分类模型，{metric_name}: {val_metric:.4f}")
            else:
                epochs_since_improvement += 1
                print(f"未改进，连续 {epochs_since_improvement} 个epoch")

        else:  # 回归任务
            val_metrics = evaluate(model, val_loader, device)
            val_metric = val_metrics['mse']  # 使用MSE作为主要指标
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"训练损失: {train_loss:.4f} | 验证MSE: {val_metric:.4f} | "
                  f"验证MAE: {val_metrics['mae']:.4f}")

            # 检查是否是最佳模型
            if val_metric < best_metric:
                best_metric = val_metric
                epochs_since_improvement = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"保存新的最佳回归模型，MSE: {val_metric:.4f}")
            else:
                epochs_since_improvement += 1
                print(f"未改进，连续 {epochs_since_improvement} 个epoch")

        # 提前停止检查
        if epochs_since_improvement >= 10:  # 10个epoch没有改进则停止
            print(f"连续10个epoch验证指标没有改进，提前停止训练")
            break

        # 学习率调度
        scheduler.step(val_metric)

    # 加载最佳模型
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        print(f"加载最佳模型权重")

    print(f"训练完成！最佳{metric_name}: {best_metric:.4f}")
    print(f"最佳模型已保存至: {best_model_path}")
    return model

def evaluate_model(model, dataloader, task_type):
    model.eval()
    device = next(model.parameters()).device

    if task_type == 'classification':
        correct = 0
        total = 0
        all_predictions = []
        all_true_labels = []
    else:  # 回归任务
        all_preds = []
        all_targets = []

    with torch.no_grad():
        for batch_data, batch_labels in dataloader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_data)

            if task_type == 'classification':
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == batch_labels).sum().item()
                total += batch_labels.size(0)
                all_predictions.extend(predicted.cpu().numpy())
                all_true_labels.extend(batch_labels.cpu().numpy())
            else:  # 回归任务
                all_preds.append(outputs.cpu())
                all_targets.append(batch_labels.cpu())

    if task_type == 'classification':
        accuracy = correct / total if total > 0 else 0.0
        return accuracy, all_predictions, all_true_labels
    else:
        if all_preds:
            all_preds = torch.cat(all_preds)
            all_targets = torch.cat(all_targets)
            return None, all_preds.numpy(), all_targets.numpy()
        return None, np.array([]), np.array([])

In [ ]:
class CSIData:
    """
    A class to represent a CSI data object.
    """

    def __init__(self, file_name: str):
        """
        Initializes the CSIData object with the provided data.

        :param file_name: The name of a file with raw data.
        """
        self.file_name = file_name
        self.frames = []

    def add_frame(self, frame):
        """
        Adds a CSI frame to the CSIData object.

        :param frame: The CSI frame to be added.
        """
        self.frames.append(frame)


class CSIFrame:
    slot = []

    def __init__(self):
        pass

    def __len__(self):
        return len(self.slot)


class BfeeFrame(CSIFrame):
    """
    Represents a WiDAR Bfee frame.
    """

    def __init__(self, timestamp_low, bfee_count, n_rx, n_tx,
                 rssi_a, rssi_b, rssi_c, noise, csi_array, agc, antenna_sel, fake_rate):
        super().__init__()
        self.timestamp_low = timestamp_low
        self.bfee_count = bfee_count
        self.n_rx = n_rx
        self.n_tx = n_tx
        self.rssi_a = rssi_a
        self.rssi_b = rssi_b
        self.rssi_c = rssi_c
        self.noise = noise
        self.csi_array = csi_array
        self.agc = agc
        self.antenna_sel = antenna_sel
        self.fake_rate = fake_rate


In [ ]:
class CSIDataset(Dataset):
    def __init__(self, tensor_list, labels=None):
        self.tensors = tensor_list
        self.labels = labels
        self.has_labels = labels is not None

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        x = self.tensors[idx]
        if isinstance(x, np.ndarray):  # numpy 转 tensor
            x = torch.from_numpy(x)
        x = x.float()
        if self.has_labels:
            return x, self.labels[idx]
        return x

def csi_collate_fn(batch):
    csi_samples, labels = zip(*batch)

    max_time = max(s.shape[0] for s in csi_samples)
    max_freq = max(s.shape[1] for s in csi_samples)
    max_ant = max(s.shape[2] for s in csi_samples)
    num_channels = csi_samples[0].shape[3]  # 处理最后一维 (2 通道)

    padded_csi = torch.zeros(
        len(csi_samples), max_time, max_freq, max_ant, num_channels, dtype=torch.float32
    )

    for i, csi in enumerate(csi_samples):
        time, freq, ant, ch = csi.shape
        padded_csi[i, :time, :freq, :ant, :ch] = csi

    labels = torch.tensor(labels, dtype=torch.long)
    return padded_csi, labels


In [ ]:
class BfeeProcessor():
    def process(self, data_list: List[CSIData], **kwargs):
        sub_type = kwargs.get('sub_type', '')

        # 这段代码把一个csi数据文件整理成一个tensor，Val acc保持在0.6不变，不能这么做
        all_tensors = []
        all_labels = []
        for csi_data in data_list:
            label = int(self.parse_file_info_from_filename(csi_data.file_name, sub_type)[1]) - 1

            # 1. 按时间戳排序帧
            sorted_frames = sorted(csi_data.frames, key=lambda frame: frame.timestamp_low)

            # 2. 处理当前文件的所有帧
            frame_tensors = []  # 存储当前文件所有帧的张量
            for frame in sorted_frames:
                csi_tensor = torch.from_numpy(frame.csi_array.astype(np.complex64))
                # 去掉其中的发射天线维度 由于widar固定为1 故直接使用squeeze()
                csi_tensor = csi_tensor.squeeze()
                csi_amp = torch.abs(csi_tensor)
                csi_db = 20 * torch.log10(csi_amp + 1e-6)
                csi_db_norm = (csi_db - csi_db.mean()) / (csi_db.std() + 1e-6)
                frame_tensors.append(csi_db_norm)  # 形状: [子载波, rx]

            # 3. 将帧堆叠成三维张量
            if frame_tensors:
                # 在维度0（帧维度）堆叠
                file_tensor = torch.stack(frame_tensors, dim=0)  # 形状: [帧数, 子载波, rx]
                all_tensors.append(file_tensor)
                all_labels.append(label)

        return all_tensors, all_labels

        # 这段代码把每个frame作为一个tensor 存在Val acc快速达到0.99的问题
        # all_tensors = []
        # all_labels = []
        # for csi_data in data_list:
        #     label = int(self.parse_file_info_from_filename(csi_data.file_name, sub_type)[1]) - 1
        #     for frame in csi_data.frames:
        #         csi_tensor = torch.from_numpy(frame.csi_array.astype(np.complex64))
        #         # permute: [sc, rx, tx] -> [sc, tx, rx]
        #         csi_permuted = csi_tensor.permute(0, 2, 1)  # shape [sc, tx, rx]
        #         # 取振幅
        #         csi_amp = torch.abs(csi_permuted)
        #         # dB
        #         csi_db = 20 * torch.log10(csi_amp + 1e-6)
        #         # 标准化
        #         csi_db_norm = (csi_db - csi_db.mean()) / (csi_db.std() + 1e-6)
        #         # (1, sc, tx, rx)
        #         csi_db_norm = csi_db_norm.unsqueeze(0)  # 1 channel
        #         all_tensors.append(csi_db_norm)
        #         all_labels.append(label)
        # return all_tensors, all_labels


    def parse_file_info_from_filename(self, f_name, task_type):
        """
        Parse the file name and return relevant information based on task_type.
        For Gesture Recognition: "id-a-b-c-d-Rx.dat"
        For Gait Recognition: "user1-1-1-r1.dat" => "user1"
        """
        base = os.path.splitext(os.path.basename(f_name))[0]

        if task_type == 'Gesture Recognition':
            m = re.match(r'user(\d+)-(\d+)-(\d+)-(\d+)-(\d+)-r(\d+)', base)
            if m:
                user_id = int(m.group(1))
                gesture_type = int(m.group(2))
                torso_position = int(m.group(3))
                orientation = int(m.group(4))
                data_serial = int(m.group(5))
                receiver_number = int(m.group(6))
                return user_id, gesture_type, torso_position, orientation, data_serial, receiver_number
            else:
                print(f"[Warning] Skipping file {f_name}: Invalid format for Gesture Recognition.")

        elif task_type == 'Activity Recognition':
            # Parse for Gait Recognition (pattern "user3-1-1-1-1-r1.dat")
            m = re.search(r'user(\d+)', base, re.IGNORECASE)
            if m:
                user_id = int(m.group(1))
                return user_id, None, None, None, None, None
            else:
                print(f"[Warning] Skipping file {f_name}: Invalid format for Activity Recognition.")

        else:
            print(f"[Error] Unknown task type: {task_type}")


In [ ]:
SIZE_STRUCT = struct.Struct(">H").unpack
CODE_STRUCT = struct.Struct("B").unpack

HEADER_STRUCT = struct.Struct("<LHHBBBBBbBBHH").unpack
VALID_BEAMFORMING_MEASUREMENT = 0xBB


class BfeeReader():
    """
    Reader for WiDAR bfee files.
    """

    def __init__(self, file_path: str):
        super().__init__()
        self.file_path = file_path

    @classmethod
    def can_read(cls, file_path: str) -> bool:
        """
        Check if the reader can read the file at the given path.

        Args:
            path (str): The path to the file to check.

        Returns:
            bool: True if the reader can read the file, False otherwise.
            :param self:
            :param file_path: The path to the file to check
        """
        with open(file_path, 'rb') as f:
            data = f.read(3)
        if len(data) < 3:
            return False
        size = SIZE_STRUCT(data[:2])[0]
        code = CODE_STRUCT(data[2:3])[0]
        if size < 20 or code != VALID_BEAMFORMING_MEASUREMENT:
            return False
        return file_path.endswith('.dat')

    def read_file(self, file_path: str) -> CSIData:
        """
        参考 Intel 5300 read_bfee.c/read_bfee_new.c 的逻辑，对单条 BFEE payload 做解析。
        返回 bfee_dict: {
          'timestamp_low': int,
          'Nrx': int, 'Ntx': int,
          'rssi_a': int, 'rssi_b': int, 'rssi_c': int,
          'noise': int(有符号),
          'csi': shape=(30, Nrx, Ntx), dtype=complex64,
          ...
        }
        """
        file_name = os.path.basename(file_path)
        ret_data = CSIData(file_name)

        with open(file_path, 'rb') as f:
            filesize = os.fstat(f.fileno()).st_size
            cur = 0
            while (cur + 3) < filesize:
                hdr = f.read(3)
                if len(hdr) < 3: break
                field_len = (hdr[0] << 8) | hdr[1]
                code = hdr[2]
                cur += 3
                if code == 0xBB:
                    payload = f.read(field_len - 1)
                    cur += (field_len - 1)
                    if len(payload) < (field_len - 1):
                        break
                    frame = self.parse_bfee_record(payload)
                    if frame is not None:
                        ret_data.add_frame(frame)
                else:
                    f.seek(field_len - 1, 1)
                    cur += (field_len - 1)
        print(f"[Info] {file_name}: B_FEE records={len(ret_data.frames)}")
        return ret_data

    @staticmethod
    def parse_bfee_record(payload: bytes):
        """
        参考 Intel 5300 read_bfee.c/read_bfee_new.c 的逻辑，对单条 BFEE payload 做解析。
        返回 bfee_dict: {
          'timestamp_low': int,
          'Nrx': int, 'Ntx': int,
          'rssi_a': int, 'rssi_b': int, 'rssi_c': int,
          'noise': int(有符号),
          'csi': shape=(30, Nrx, Ntx), dtype=complex64,
          ...
        }
        """
        if len(payload) < 20:
            return None

        timestamp_low = (payload[0] |
                         (payload[1] << 8) |
                         (payload[2] << 16) |
                         (payload[3] << 24)) & 0xffffffff
        bfee_count = (payload[4] | (payload[5] << 8)) & 0xffff

        Nrx = payload[8]
        Ntx = payload[9]
        rssi_a = payload[10]
        rssi_b = payload[11]
        rssi_c = payload[12]
        noise = struct.unpack('b', payload[13:14])[0]
        agc = payload[14]
        antenna_sel = payload[15]
        csi_len = (payload[16] | (payload[17] << 8)) & 0xffff
        fake_rate = (payload[18] | (payload[19] << 8)) & 0xffff

        calc_len = (30 * (Nrx * Ntx * 8 * 2 + 3) + 7) // 8
        if csi_len != calc_len: return None
        if len(payload) < (20 + csi_len): return None

        csi_bytes = payload[20: 20 + csi_len]
        csi_array = np.zeros((30, Nrx, Ntx), dtype=np.complex64)

        bit_index = 0

        def get_bit(pos):
            byte_i = pos // 8
            if byte_i >= len(csi_bytes):
                return 0
            shift = pos % 8
            return (csi_bytes[byte_i] >> shift) & 0x1

        def get_bits_u8(pos):
            val = 0
            for b in range(8):
                val |= (get_bit(pos + b) << b)
            return val

        for sc_idx in range(30):
            bit_index += 3  # skip pilot
            for j in range(Nrx * Ntx):
                real8 = get_bits_u8(bit_index)
                imag8 = get_bits_u8(bit_index + 8)
                bit_index += 16
                if real8 & 0x80: real8 -= 256
                if imag8 & 0x80: imag8 -= 256
                rx_i = j % Nrx
                tx_i = j // Nrx
                csi_array[sc_idx, rx_i, tx_i] = np.complex64(real8 + 1j * imag8)
        return BfeeFrame(timestamp_low, bfee_count, Nrx, Ntx, rssi_a, rssi_b, rssi_c,
                         noise, csi_array, agc, antenna_sel, fake_rate)


In [ ]:
import numpy as np
import pywt
import time
from scipy.stats import linregress


def batch_preprocess(csi_data_list, denoise=True, normalize=True, phase_correction=True):
    print(f"Starting batch preprocessing for {len(csi_data_list)} samples...")
    start_time = time.time()
    processed_data = []

    for i, csi_data in enumerate(csi_data_list):
        try:
            processed = preprocess_csi(
                csi_data,
                denoise=denoise,
                normalize=normalize,
                phase_correction=phase_correction
            )
            processed_data.append(processed)
        except Exception as e:
            print(f"Preprocess failed on sample {i}: {e}")
            processed_data.append(None)
        if (i + 1) % 10 == 0 or (i + 1) == len(csi_data_list):
            print(f"Processed {i + 1}/{len(csi_data_list)} samples")

    end_time = time.time()
    print(f"Batch preprocessing completed in {end_time - start_time:.2f} seconds")
    return [d for d in processed_data if d is not None]  # 去除 None

def preprocess_csi(csi_data, denoise=True, normalize=True, phase_correction=True):
    csi_data = np.array(csi_data, dtype=np.complex64, copy=True)

    if phase_correction:
        time_idx = np.arange(csi_data.shape[0])
        for sc in range(csi_data.shape[1]):
            for rx in range(csi_data.shape[2]):
                phase = np.unwrap(np.angle(csi_data[:, sc, rx]), discont=np.pi)
                if len(phase) < 5 or np.std(phase) < 1e-3:
                    continue
                try:
                    slope, intercept, *_ = linregress(time_idx, phase)
                    if not np.isfinite(slope):
                        continue
                    csi_data[:, sc, rx] *= np.exp(-1j * slope * time_idx)
                except Exception as e:
                    print(f"Phase correction failed: sc={sc} rx={rx} {e}")
                    continue

    amplitude = np.abs(csi_data)
    phase = np.angle(csi_data)

    if denoise:
        def wavelet_denoise(channel):
            try:
                L = len(channel)
                if L < 8:
                    return channel
                max_level = pywt.dwt_max_level(L, pywt.Wavelet('db4').dec_len)
                level = min(2, max_level)
                if level < 1:
                    level = 1
                if L < 16:
                    level = 1
                coeffs = pywt.wavedec(channel, 'db4', level=level)
                sigma = np.median(np.abs(coeffs[-level])) / 0.6745
                threshold = sigma * np.sqrt(2 * np.log(L))
                denoised_coeffs = [coeffs[0]] + [
                    pywt.threshold(c, threshold, 'soft') for c in coeffs[1:]
                ]
                denoised = pywt.waverec(denoised_coeffs, 'db4')
                return denoised[:L]
            except Exception as e:
                print(f"Denoising failed: {e}")
                return channel

        for rx in range(amplitude.shape[2]):
            for sc in range(amplitude.shape[1]):
                amp_seq = amplitude[:, sc, rx]
                amplitude[:, sc, rx] = wavelet_denoise(amp_seq)

    if normalize:
        reshaped = amplitude.transpose(1, 0, 2).reshape(amplitude.shape[1], -1)
        amp_min = np.min(reshaped, axis=1, keepdims=True)
        amp_max = np.max(reshaped, axis=1, keepdims=True)
        amp_min = amp_min.reshape(1, -1, 1)
        amp_max = amp_max.reshape(1, -1, 1)
        denominator = amp_max - amp_min
        denominator[denominator < 1e-6] = 1.0
        amplitude = (amplitude - amp_min) / (denominator + 1e-10)

    processed_csi = amplitude * np.exp(1j * phase)
    return np.stack([np.real(processed_csi), np.imag(processed_csi)], axis=-1)  # shape: [T, S, R, 2]


In [ ]:
"""
执行位置
"""


folder_path = Path(folder_path)
if not folder_path.exists() or not folder_path.is_dir():
    raise ValueError(f"无效的文件夹路径: {folder_path}")
files = [f for f in folder_path.rglob("*") if f.is_file() and "truth" not in f.name]
if not files:
    raise IOError(f"文件夹 {folder_path} 中没有文件")

# 使用第一个文件确定主格式
sample_file = files[0]
reader = BfeeReader(sample_file)
sample_frame = reader.read_file(sample_file).frames[0]
processor = BfeeProcessor()

print(f"检测到主文件格式: {type(reader).__name__}\n")

print(f"开始处理 {len(files)} 个文件...\n")

csi_data_list = []
# 处理所有文件
for file_path in files:
    try:
        csi_data = reader.read_file(str(file_path))
        csi_data_list.append(csi_data)

        print(f"√ 已处理: {file_path.name}\n")

    except Exception as e:
        print(f"× 处理失败 {file_path.name}: {str(e)}\n")

print(f"处理完成! 共处理 {len(files)} 个文件\n")


res = processor.process(csi_data_list, folder_path=folder_path,sub_type=sub_type, final_fs=final_fs)


# 处理csi
print("Preprocessing CSI data...\n")
csi_data_list = res[0]
labels = res[1]
# 这一步还是成功的 print(csi_data_list)
# Ensure labels are zero-indexed
unique_labels = sorted(list(set(labels)))
label_map = {label: i for i, label in enumerate(unique_labels)}
zero_indexed_labels = [label_map[label] for label in labels]

processed_csi_data = batch_preprocess(
    csi_data_list,
    denoise=True,
    normalize=True,
    phase_correction=True
)
print("process dataset success\n")
train_data, val_data, train_labels, val_labels = train_test_split(
    processed_csi_data, zero_indexed_labels, test_size=0.2, random_state=42
)
print(f"Train samples: {len(train_data)}, Validation samples: {len(val_data)}\n")
train_dataset = CSIDataset(train_data, train_labels)
val_dataset = CSIDataset(val_data, val_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=csi_collate_fn,
    num_workers=8,  # 使用多进程加载
    pin_memory=True  # 加速GPU传输
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=csi_collate_fn,
    num_workers=8,
    pin_memory=True
)

num_classes = len(unique_labels)
if task_type == 'classification':
    model = AdaptiveCSIModel(task_type, num_classes=num_classes)
else:
    model = None  # for more task_type in the future
print("Model architecture:")
print(model)

print("\nStarting training...")
trained_model = train_model(
    model,
    train_loader,
    val_loader
)

torch.save(trained_model.state_dict(), "./sdp/results/saved_models/final_csi_model.pth")
print("Final model saved to /sdp/results/saved_models/final_csi_model.pth\n")

Streaming output truncated to the last 5000 lines.
[Info] user3-4-1-3-3-r3.dat: B_FEE records=1601
√ 已处理: user3-4-1-3-3-r3.dat

[Info] user3-4-1-3-3-r5.dat: B_FEE records=1599
√ 已处理: user3-4-1-3-3-r5.dat

[Info] user3-4-1-3-3-r6.dat: B_FEE records=1602
√ 已处理: user3-4-1-3-3-r6.dat

[Info] user3-4-1-3-4-r1.dat: B_FEE records=1660
√ 已处理: user3-4-1-3-4-r1.dat

[Info] user3-4-1-3-4-r2.dat: B_FEE records=1660
√ 已处理: user3-4-1-3-4-r2.dat

[Info] user3-4-1-3-4-r3.dat: B_FEE records=1662
√ 已处理: user3-4-1-3-4-r3.dat

[Info] user3-4-1-3-4-r4.dat: B_FEE records=1661
√ 已处理: user3-4-1-3-4-r4.dat

[Info] user3-4-1-3-4-r5.dat: B_FEE records=1660
√ 已处理: user3-4-1-3-4-r5.dat

[Info] user3-4-1-3-4-r6.dat: B_FEE records=1661
√ 已处理: user3-4-1-3-4-r6.dat

[Info] user3-4-1-3-5-r1.dat: B_FEE records=1513
√ 已处理: user3-4-1-3-5-r1.dat

[Info] user3-4-1-3-5-r2.dat: B_FEE records=1513
√ 已处理: user3-4-1-3-5-r2.dat

[Info] user3-4-1-3-5-r3.dat: B_FEE records=1515
√ 已处理: user3-4-1-3-5-r3.dat

[Info] user3-4-1-3-5-r4.d

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import pickle


# ============ 测试集评估 ============
print("\n" + "="*50)
print("开始测试集评估")
print("="*50)

# 评估函数
def evaluate_model(model, dataloader, task_type):
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_true_labels = []
    device = next(model.parameters()).device # Get the device the model is on

    with torch.no_grad():
        for batch_data, batch_labels in dataloader:
            batch_data = batch_data.to(device) # Move data to the same device as the model
            batch_labels = batch_labels.to(device) # Move labels to the same device as the model

            outputs = model(batch_data)

            if task_type == 'classification':
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == batch_labels).sum().item()
                all_predictions.extend(predicted.cpu().numpy())
                all_true_labels.extend(batch_labels.cpu().numpy())

            total += batch_labels.size(0)

    accuracy = correct / total
    return accuracy, all_predictions, all_true_labels

# 使用训练好的模型进行评估
test_accuracy, predictions, true_labels = evaluate_model(
    trained_model, val_loader, task_type
)

# 打印评估结果
print(f"\n测试集准确率: {test_accuracy:.4f}")

# 分类报告
print("\n分类报告:")
print(classification_report(true_labels, predictions))

# 混淆矩阵
conf_matrix = confusion_matrix(true_labels, predictions)
print("\n混淆矩阵:")
print(conf_matrix)

# 保存评估结果
eval_results = {
    'accuracy': test_accuracy,
    'predictions': predictions,
    'true_labels': true_labels,
    'confusion_matrix': conf_matrix
}
with open('/content/drive/MyDrive/eval_results/evaluation_results.pkl', 'wb') as f:
    pickle.dump(eval_results, f)
print("\n评估结果已保存到 /content/drive/MyDrive/eval_results/evaluation_results.pkl")